In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weight_filename, naanobot_weight_filename,
                                          skeleton_config, naanobot_config, radial_config)

In [2]:
skeleton_weight_filename = skeleton_weight_filename
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weight_filename
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [6]:
pocket_data = model.generate_pocket_data(temperature=0.3)
print(len(pocket_data))
print(type(pocket_data))

4
<class 'dict'>


In [7]:
pocket_data

{'SMILES': 'CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C',
 'Temperature': 0.3,
 '3D_skeleton': [[4.209573274789946, 14.813610879830014, -0.5892869294724622],
  [2.886580217269598, 13.922717403778986, -2.590176966467803],
  [7.064341612114732, 11.98708012364927, -2.1062091232208373],
  [6.192752864161234, 12.093276112028505, -2.104457694118446],
  [10.405480699571784, 7.579420222612508, -2.777137445702382],
  [6.143932554297379, 10.545978070406788, -3.901270844457713],
  [8.80422502958957, 8.78818304294799, -2.3468565033028885],
  [9.969240524011193, 6.878647125595166, -2.252932187388627],
  [10.477258601088206, 4.98701085086146, -2.7557954805616234],
  [9.429504009083074, 5.663386538493657, -2.810713942712328],
  [6.981802322734368, 8.145011353377964, -2.1733684899370553],
  [5.377433081376344, 8.079613822837752, -3.1866162605057644],
  [6.151751018731435, 7.08220796991183, -2.3751984108353676],
  [3.0738554870050345, 7.419512670914529, -1.8066303357412208],
  [4.667445710460162

In [8]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [9]:
# GENERATION QUALITY ASSESSMENTS

In [19]:
import torch
import torch.nn.functional as F
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=50):
    model.eval()
    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]
    aa_chain = ""
    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0).to(map4_enc.device)
        output, _ = model.forward(naano_X, map4_enc)

        # print raw predicted vector
        print(f"\nPosition {i}:")
        print(f"  raw output: {output[0].detach().cpu().numpy().round(2)}")

        # print distances to all amino acids
        vector = output[0].detach().float()
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = F.mse_loss(vector,n_v_t)


        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        print(f"  top 3: {sorted_distances[:20]}")

        # update context
        aa_id = model.naano_module.approx_id(output[0], temperature=0.1)
        bioch_context = bioch_context[1:]+[model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:]+[coordinate]
        aa_chain += aa_id
        print(aa_chain)


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
sph_coordinates = model._pocket_sph_skeleton()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=sph_coordinates)


Position 0:
  raw output: [ 0.46 -0.87 -0.61 -0.26 -1.06 -0.32 -1.07  0.08  0.69  0.63  0.65 -0.69
 -0.06 -0.7  -0.02]
  top 3: [('G', tensor(0.6306)), ('T', tensor(0.6387)), ('S', tensor(0.6406)), ('A', tensor(0.6524)), ('D', tensor(0.6578)), ('C', tensor(0.6671)), ('E', tensor(0.6764)), ('N', tensor(0.6985)), ('V', tensor(0.7046)), ('Q', tensor(0.7183)), ('M', tensor(0.7431)), ('H', tensor(0.7527)), ('L', tensor(0.7599)), ('I', tensor(0.7617)), ('P', tensor(0.7836)), ('F', tensor(0.7853)), ('Y', tensor(0.7861)), ('W', tensor(0.9229)), ('K', tensor(0.9437)), ('R', tensor(1.1013))]
D

Position 1:
  raw output: [ 0.34 -1.25 -0.68 -0.02 -0.88 -0.28 -0.81  0.19  0.42  0.9   0.64 -0.93
  0.07 -0.39 -0.21]
  top 3: [('G', tensor(0.6267)), ('D', tensor(0.6344)), ('T', tensor(0.6527)), ('C', tensor(0.6614)), ('E', tensor(0.6688)), ('S', tensor(0.6738)), ('A', tensor(0.6787)), ('V', tensor(0.7100)), ('N', tensor(0.7140)), ('H', tensor(0.7357)), ('M', tensor(0.7457)), ('Q', tensor(0.7497)), ('